# Session 5 — Your turn: Scaling and data

Fit a scaling law to measurements, then use a fitted law to split a compute budget between model size and data.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from scipy.optimize import curve_fit

rng = np.random.default_rng(2)
N = np.array([1e5, 3e5, 1e6, 3e6, 1e7, 3e7, 1e8])                       # parameters of seven trained models
loss = 1.5 + 60 * N ** -0.25 + rng.normal(0, 0.004, len(N))             # their validation losses (a made-up law with noise)
print(np.round(loss, 3))
from aij_check import check, close, summary

[4.875 4.062 3.396 2.932 2.574 2.315 2.099]


## Task 1: Fit a power law

Fit `loss = a * N**(-b) + c` to the seven points with `scipy.optimize.curve_fit` (start from `p0=[50, 0.3, 1]`) and store the fitted `a`, `b`, `c` in `a_fit`, `b_fit`, `c_fit`.
Then use the law to predict the loss of a model with 1e9 parameters in `prediction_1e9`, and set `irreducible` to the part of the loss that more parameters can never remove.

In [2]:
law = lambda N, a, b, c: a * N ** -b + c
(a_fit, b_fit, c_fit), _ = curve_fit(law, N, loss, p0=[50, 0.3, 1], maxfev=20000)
prediction_1e9 = law(1e9, a_fit, b_fit, c_fit)
irreducible = c_fit
print(a_fit, b_fit, c_fit, prediction_1e9)

61.1952270481327 0.25206404023838147 1.5139781595547686 1.843695044641905


In [3]:
check("the exponent is near 0.25", lambda: 0.2 < b_fit < 0.3)
check("the floor is near 1.5", lambda: 1.3 < c_fit < 1.7)
check("the law reproduces the measurements to within 0.02", lambda: np.abs(law(N, a_fit, b_fit, c_fit) - loss).max() < 0.02)
check("the prediction for 1e9 parameters is below the 1e8 measurement and above the floor", lambda: c_fit < prediction_1e9 < loss[-1])
check("irreducible is the constant term", lambda: irreducible == c_fit)
summary()

PASS  the exponent is near 0.25
PASS  the floor is near 1.5
PASS  the law reproduces the measurements to within 0.02
PASS  the prediction for 1e9 parameters is below the 1e8 measurement and above the floor
PASS  irreducible is the constant term

5 of 5 checks passed.  Well done.


**Why it works.** On a log scale the reducible part of the loss falls on a straight line, so a few small runs predict a bigger one, and the constant c says where adding parameters stops helping. The prediction is an extrapolation: it is only as good as the assumption that the same law keeps holding.

## Task 2: Split a compute budget

A model with `N` parameters trained on `D` tokens costs `C = 6 * N * D` operations, and the fitted law is `L(N, D) = 1.69 + 406.4 / N**0.34 + 410.7 / D**0.28`.
For a budget `C = 1e21` try the model sizes `np.logspace(7, 11, 400)`, set `D = C / (6 * N)` for each, and store the size with the lowest loss in `best_N`, its data in `best_D`, its loss in `best_loss`,
and `tokens_per_parameter = best_D / best_N`.

In [4]:
C = 1e21
sizes = np.logspace(7, 11, 400)
L = lambda N, D: 1.69 + 406.4 / N ** 0.34 + 410.7 / D ** 0.28
losses = L(sizes, C / (6 * sizes))
best = int(losses.argmin())
best_N, best_D, best_loss = sizes[best], C / (6 * sizes[best]), losses[best]
tokens_per_parameter = best_D / best_N
print(best_N, best_D, best_loss, tokens_per_parameter)

1843589221.641486 90403363563.96725 2.3288863326525653 49.03660886207305


In [5]:
check("the best size is inside the searched range, not at an edge", lambda: 1e7 < best_N < 1e11)
check("best_D uses the whole budget", lambda: close(6 * best_N * best_D, 1e21, 1e8))
check("no other size does better", lambda: all(L(n, 1e21 / (6 * n)) >= best_loss - 1e-12 for n in np.logspace(7, 11, 400)))
check("the best model sees tens of tokens per parameter (between 20 and 80)", lambda: 20 < tokens_per_parameter < 80)
summary()

PASS  the best size is inside the searched range, not at an edge
PASS  best_D uses the whole budget
PASS  no other size does better
PASS  the best model sees tens of tokens per parameter (between 20 and 80)

4 of 4 checks passed.  Well done.


**Why it works.** For a fixed budget a bigger model means fewer tokens. The loss is lowest where the two terms balance, at tens of tokens per parameter (about 49 with these constants; published fits give figures from about 20 upwards, depending on the constants): models trained with far fewer tokens per parameter were too large for the data they saw.